# 🎬 Video Hub — GPU Backend for Render App

این نوت‌بوک یک **سرور GPU** روی Colab بالا میاره که اپ Render شما بهش وصل میشه.

## 🔥 چطور کار میکنه
1. سلول‌ها رو به ترتیب اجرا کن
2. یک آدرس `https://xxxxx.ngrok-free.app` میگیری
3. اون آدرس رو در اپ Render خودت (صفحه `/video/`) وارد کن
4. تمام! از موبایل هم میتونی اپ Render رو باز کنی و ویدیو تولید کنی

> ⚠️ **حتماً Runtime رو روی T4 GPU بذار**: Runtime → Change runtime type → T4 GPU (رایگان)

## گام ۱: کلون ریپو و نصب

In [ ]:
!git clone https://github.com/farzadabbasi617-star/game-lead-finder.git /content/repo 2>/dev/null || (cd /content/repo && git pull)
%cd /content/repo
!pip install -q fastapi uvicorn python-multipart pyngrok httpx
!pip install -q diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece protobuf Pillow
print('✅ نصب کامل شد')

## گام ۲: چک GPU

In [ ]:
import torch
assert torch.cuda.is_available(), '❌ GPU پیدا نشد! Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## گام ۳: تنظیم ngrok

توکن رایگان از https://dashboard.ngrok.com/get-started/your-authtoken بگیر و اینجا بذار:

In [ ]:
NGROK_TOKEN = '2abc...def'  # 👈 توکن ngrok خودت رو اینجا بذار

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

# قطع تونل‌های قبلی (اگر باشه)
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
print('✅ ngrok آماده است')

## گام ۴: راه‌اندازی GPU Backend Server

In [ ]:
import threading, time, uvicorn, sys
sys.path.insert(0, '/content/repo')
sys.path.insert(0, '/content/repo/colab')

from gpu_backend_server import app

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)

# باز کردن تونل ngrok
public = ngrok.connect(8000)
BACKEND_URL = public.public_url

print('=' * 60)
print(f'🎉 GPU Backend فعال شد!')
print('=' * 60)
print(f'🔗 آدرس این backend:')
print(f'   {BACKEND_URL}')
print()
print(f'📋 حالا برو به اپ Render خودت:')
print(f'   https://YOUR-APP.onrender.com/video/')
print(f'   و این آدرس رو در فیلد "GPU Backend" وارد کن')
print('=' * 60)
print()
print(f'🧪 تست: {BACKEND_URL}/health')
print(f'📖 Docs: {BACKEND_URL}/docs')

## گام ۵: تست دستی (اختیاری)

برای اطمینان که backend کار میکنه:

In [ ]:
import httpx
print('=== Health ===')
print(httpx.get(f'{BACKEND_URL}/health').json())

## ⚠️ نگه‌داری اتصال زنده

Colab بعد از **۹۰ دقیقه بی‌کاری** disconnect میشه. برای اینکه فعال بمونه، این سلول رو اجرا کن (هر ۵ دقیقه ping میزنه):

In [ ]:
import time, httpx
print('🔄 در حال نگه‌داری اتصال... (Ctrl+C برای توقف)')
while True:
    try:
        r = httpx.get(f'{BACKEND_URL}/health', timeout=10)
        print(f'[{time.strftime("%H:%M:%S")}] ✅ alive - {r.json().get("gpu", "?")}')
    except Exception as e:
        print(f'[{time.strftime("%H:%M:%S")}] ❌ {e}')
    time.sleep(300)

## 📝 نکات مهم

- **Colab رایگان**: هر جلسه تا ۱۲ ساعت GPU T4 میده. بعد باید دوباره اجرا کنی.
- **ngrok رایگان**: هر بار که Colab قطع بشه، آدرس عوض میشه. باید آدرس جدید رو در اپ Render وارد کنی.
- **مدل‌های سبک برای T4** (~16GB): CogVideoX-2B، Wan 2.1 1.3B، LTX-Video، AnimateDiff، Zeroscope
- **مدل‌های سنگین**: HunyuanVideo، Mochi، Wan 14B → نیاز به Colab Pro + A100